# 1. Dataset

In [1]:
import torch
from torch.utils.data import Dataset
import numpy as np
import cv2
from torch import randint
import os
import random
from torchvision import transforms
import pandas as pd



def seed_everything(seed: int):
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = True


mean = [0.26524142  , 0.26524142 ,0.26524142 ]
std = [0.04526951 , 0.04526951 , 0.04526951 ]
data_transforms = {
    'training': transforms.Compose([
        transforms.ToPILImage(),
        transforms.RandomHorizontalFlip(p=0.3),
        transforms.RandomApply(torch.nn.ModuleList([transforms.ColorJitter(), ]), p=0.3),
        transforms.RandomApply(torch.nn.ModuleList([transforms.GaussianBlur(kernel_size=3), ]), p=0.3),
        transforms.ToTensor(),
        transforms.Normalize(mean, std)
    ]),
    'valid': transforms.Compose([
        transforms.ToPILImage(),
        transforms.ToTensor(),
        transforms.Normalize(mean, std)
    ]),
    'test': transforms.Compose([
        transforms.ToPILImage(),
        transforms.ToTensor(),
    ]),
}


class MammoCompDataset(Dataset):
    def __init__(self,
                 data_path="../VinDr_Mammo/physionet.org/files/vindr-mammo/1.0.0/images_png/",
                 metadata="../VinDr_Mammo/physionet.org/files/vindr-mammo/1.0.0/breast-level_annotations1.csv",
                 phase="train",
                 mode="binary_contrastive",
                 transform=None,
                 datalen=100,
                 certain=True,
                 seed=None):
        self.phase = phase
        self.datalen = datalen  # Number of image pairs for training/testing
        self.certain = certain
        self.mode = mode
        self.data_path = data_path
        if (seed):
            seed_everything(seed)

        self.transform = data_transforms[self.phase] if (transform == None) else transform
        data = pd.read_csv(metadata)
        self.data = data.loc[data['split'] == phase].reset_index()
        self.birads = []
        for i in range(1, 6):
            self.birads.append(self.data.loc[self.data['breast_birads'] == f'BI-RADS {i}'])
        self.name_of_classes = [1, 2, 3, 4, 5]
        self.len_of_classes = [len(self.birads[i].index) for i in range(5)]
        self.paths1 = []
        self.paths2 = []
        self.listi1 = []
        self.listi2 = []
        self.complabels = []
        curlen = 0
        self.imagesinclass0 = self.birads[0]
        seed_everything(seed)
        while (curlen < self.datalen):
            if (mode == 'multiclass_contrastive'):
                if (random.randint(0, 1) == 0):
                    i1 = random.randint(0, 4)
                    i2 = random.randint(0, 4)
                else:
                    i1 = random.randint(0, 4)
                    i2 = i1
                # pickimageA = randint(0, lenofclass[random_pick_2class[0]], (1,))
                self.listi1.append(i1)
                self.listi2.append(i2)
                self.paths1.append(self.get_path(self.birads[i1], randint(0, self.len_of_classes[i1], (1,))[0]))
                self.paths2.append(self.get_path(self.birads[i2], randint(0, self.len_of_classes[i2], (1,))[0]))
                
                self.complabels.append((i1==i2)*1)
                curlen = curlen + 1
            elif mode == 'binary_contrastive':
                modee = random.randint(0, 3)
                if (modee == 0):
                    i1 = 0
                    i2 = 0
                elif (modee == 1):
                    i1 = random.randint(1, 4)
                    i2 = random.randint(1, 4)
                elif (modee == 2):
                    i1 = 0
                    i2 = random.randint(1, 4)
                else:
                    i2 = 0
                    i1 = random.randint(1, 4)
                # pickimageA = randint(0, lenofclass[random_pick_2class[0]], (1,))
                self.listi1.append(i1)
                self.listi2.append(i2)
                self.paths1.append(self.get_path(self.birads[i1], randint(0, self.len_of_classes[i1], (1,))[0]))
                self.paths2.append(self.get_path(self.birads[i2], randint(0, self.len_of_classes[i2], (1,))[0]))
                self.complabels.append(1 if (((i1 == 0) and (i2 == 0)) or ((i1 != 0) and (i2 != 0))) * 1 else -1)
                curlen = curlen + 1
            elif (mode == 'severity_comparison'):
                i1 = random.randint(1, 4)
                i2 = random.randint(1, 4)
                # pickimageA = randint(0, lenofclass[random_pick_2class[0]], (1,))
                self.listi1.append(i1)
                self.listi2.append(i2)
                self.paths1.append(self.get_path(self.birads[i1], randint(0, self.len_of_classes[i1], (1,))[0]))
                self.paths2.append(self.get_path(self.birads[i2], randint(0, self.len_of_classes[i2], (1,))[0]))
                self.complabels.append(((i1 > i2)) * 1)
                curlen = curlen + 1
            elif (mode == 'preference_contrastive'):
                if (random.randint(0, 5) > 4):
                    i1 = random.randint(1, 4)
                    i2 = i1
                else:
                    i1 = random.randint(1, 4)
                    i2 = random.randint(1, 4)
                # pickimageA = randint(0, lenofclass[random_pick_2class[0]], (1,))
                self.listi1.append(i1)
                self.listi2.append(i2)
                self.paths1.append(self.get_path(self.birads[i1], randint(0, self.len_of_classes[i1], (1,))[0]))
                self.paths2.append(self.get_path(self.birads[i2], randint(0, self.len_of_classes[i2], (1,))[0]))
                self.complabels.append((((i1 > i2)) * 1) if (i1 != i2) else 2)
                curlen = curlen + 1
            else:
                assert False, f"No mode {mode} found, please try multiclass_contrastive or binary_contrastive"

    def get_score(self, data, index):
        birads = data['breast_birads'].iloc[index.item()]
        score = eval(birads[-1])
        return score

    def get_path(self, data, index):

        image_name = data['image_id'].iloc[index.item()]
        study_id = data['study_id'].iloc[index.item()]
        image_path = os.path.join(self.data_path, study_id + '/' + image_name + '.png')
        return (image_path)

    def __getitem__(self, index):
        imageA = cv2.imread(self.paths1[index])
        imageB = cv2.imread(self.paths2[index])

        label = self.complabels[index]

        imageA = self.transform(imageA)
        imageB = self.transform(imageB)
        if self.mode == 'severity_comparison':
            ref_img = self.get_ref_images()
            return (imageA, imageB), ref_img, label, (self.listi1[index], self.listi2[index])
        else:
            return (imageA, imageB), label, (self.listi1[index], self.listi2[index])

    def get_ref_images(self):
        ref_img = self.get_path(self.imagesinclass0, randint(0, len(self.imagesinclass0), (1,))[0])
        ref_img = cv2.imread(ref_img)
        ref_img = self.transform(ref_img)

        return ref_img

    def __len__(self):
        return self.datalen
    

# 2. Model

In [2]:
from torch import nn
def get_default_fc(in_features=2048, model='siamese1', ncriteria=10):
    if(model=='siamese1'):
        ret =  nn.Sequential(torch.nn.Linear(in_features, 256),
                                torch.nn.ReLU(),
                                torch.nn.Dropout(0.1),
                                torch.nn.Linear(256, 64),
                                torch.nn.ReLU(),
                                torch.nn.Dropout(0.1),
                                torch.nn.Linear(64, 1))
    else:
        ret =  nn.Sequential(torch.nn.Linear(in_features, 256),
                                torch.nn.ReLU(),
                                torch.nn.Dropout(0.1),
                                torch.nn.Linear(256, 64),
                                torch.nn.ReLU(),
                                torch.nn.Dropout(0.1),
                                torch.nn.Linear(64, ncriteria))
    return ret
class ResNetSimCLR(nn.Module):

    def __init__(self, base_model, out_dim):
        super(ResNetSimCLR, self).__init__()
        self.resnet_dict = {"resnet18": models.resnet18(weights='ResNet18_Weights.DEFAULT', num_classes=out_dim),
                            "resnet50": models.resnet50(weights='ResNet50_Weights.DEFAULT', num_classes=out_dim),
                            "resnet101": models.resnet101(weights='ResNet101_Weights.DEFAULT', num_classes=out_dim),
                            "densenet121": models.densenet121(weights='DenseNet121_Weights.DEFAULT', num_classes=out_dim)}

        self.backbone = self._get_basemodel(base_model)
        dim_mlp = self.backbone.fc.in_features

        # add mlp projection head
        self.backbone.fc = nn.Sequential(nn.Linear(dim_mlp, dim_mlp), nn.ReLU(), self.backbone.fc)

    def _get_basemodel(self, model_name):
        try:
            model = self.resnet_dict[model_name]
        except KeyError:
            raise InvalidBackboneError(
                "Invalid backbone architecture. Check the config file and pass one of: resnet18 or resnet50")
        else:
            return model

    def forward(self, x):
        return self.backbone(x)

In [3]:
from torchvision import models, transforms

def get_feature_extractor(feature_extractor = 'resnet50', fcnet = None, cotrain=True, ncriteria=10, model='siamese1', simclr = None):
    if(feature_extractor == 'resnet50'):    
        fextractor = models.resnet50(weights='ResNet50_Weights.DEFAULT')
        in_features = 2048
        if(simclr):
            print('load simclr resnet')
            ressimclr = ResNetSimCLR('resnet50', 1000)
            state_dict = torch.load(simclr)
            ressimclr.load_state_dict(state_dict['state_dict'])
            fextractor = ressimclr.backbone
        fextractor.fc = get_default_fc(in_features, model=model, ncriteria=ncriteria) if (fcnet == None) else fcnet
    elif(feature_extractor == 'resnet101'):    
        fextractor = models.resnet101(weights='ResNet101_Weights.DEFAULT')
        in_features = 2048
        if(simclr):
            print('load simclr resnet')
            ressimclr = ResNetSimCLR('resnet101', 1000)
            state_dict = torch.load(simclr)
            ressimclr.load_state_dict(state_dict['state_dict'])
            fextractor = ressimclr.backbone
        fextractor.fc = get_default_fc(in_features, model=model, ncriteria=ncriteria) if (fcnet == None) else fcnet
    elif(feature_extractor == 'densnet121'):
        fextractor = models.densenet121(weights='DenseNet121_Weights.DEFAULT')
        in_features = 1024
        fextractor.classifier = get_default_fc(in_features, model=model, ncriteria=ncriteria) if (fcnet == None) else fcnet
        # fextractor._modules['classifier'] = fextractor._modules.pop('classifier')
    elif(feature_extractor == 'vgg19'):
        fextractor = models.vgg19()
        fextractor.load_state_dict(torch.load('./pretrained/vgg19-dcbb9e9d.pth'))
        in_features = 25088 # https://www.geeksforgeeks.org/vgg-16-cnn-model/ length of vgg19
        fextractor.classifier = get_default_fc(in_features, model=model, ncriteria=ncriteria) if (fcnet == None) else fcnet 
        # fextractor._modules['fc'] = fextractor._modules.pop('classifier')
    elif(feature_extractor == 'vit16'):
        fextractor = models.vit_b_16()
        in_features = 768
        fextractor.load_state_dict(torch.load('./pretrained/vit_b_16-c867db91.pth'))
        fextractor.heads.head = get_default_fc(in_features, model=model, ncriteria=ncriteria) if (fcnet == None) else fcnet 
        # fextractor.classifier = get_default_fc(in_features) if (fcnet == None) else fcnet
    else:
        assert False, 'No feature extractor founded'

    for param in fextractor.parameters():
            param.requires_grad = cotrain
    if(feature_extractor == 'resnet50' or feature_extractor == 'resnet101'):        
        for param in fextractor.fc.parameters():
            param.requires_grad = True
    elif(feature_extractor == 'vit16'):
        for param in fextractor.heads.parameters():
            param.requires_grad = True
    else:
        for param in fextractor.classifier.parameters():
            param.requires_grad = True

    return fextractor

In [4]:
class SiameseNetwork101(nn.Module):
    """
    Siamese neural network
    Modified from: https://hackernoon.com/facial-similarity-with-siamese-networks-in-pytorch-9642aa9db2f7
    Siamese ResNet-101 from Pytorch library
    """ 
    def __init__(self):
        super(SiameseNetwork101, self).__init__()
        # note that resnet101 requires 3 input channels, will repeat grayscale image x3
        self.cnn1 = get_feature_extractor(feature_extractor='resnet50', cotrain=False)
        self.cnn1.fc = nn.Sequential(torch.nn.Linear(2048, 1000),
                                torch.nn.ReLU(),
                                torch.nn.Dropout(0.1),
                                torch.nn.Linear(1000, 256))
    
    def forward_once(self, x):
        output = self.cnn1(x)
        return output

    def forward(self, input1, input2):
        output1 = self.forward_once(input1)
        output2 = self.forward_once(input2)
        return output1, output2

# 3. Loss function

In [5]:
# class ContrastiveLoss(torch.nn.Module):
#     """
#     Contrastive loss function.
#     Based on: http://yann.lecun.com/exdb/publis/pdf/hadsell-chopra-lecun-06.pdf
#     Modified from: https://hackernoon.com/facial-similarity-with-siamese-networks-in-pytorch-9642aa9db2f7

#     """ 

#     def __init__(self, margin=2.0):
#         super(ContrastiveLoss, self).__init__()
#         self.margin = margin

#     def forward(self, output1, output2, label):
#         # euclidean_distance = torch.nn.functional.pairwise_distance(output1, output2)
#         # loss_contrastive = torch.mean((1-label) * torch.pow(euclidean_distance, 2) +
#         #                               (label) * torch.pow(torch.clamp(self.margin - euclidean_distance, min=0.0), 2))
        
#         cosine_distance = torch.nn.functional.cosine_similarity(output1, output2)
#         loss_contrastive = torch.mean((1-label) * torch.pow(cosine_distance, 2) +
#                                       (label) * torch.pow(torch.clamp(self.margin - cosine_distance, min=0.0), 2))
#         # loss_contrastive =  torch.nn.NLLLoss()(cosine_distance)

#         return loss_contrastive

# 4. Train pipeline

In [6]:
import torch
from torch.utils.data import DataLoader
import torch.optim as optim
from torch.optim import lr_scheduler
import os


def train_model(model, train_dataset, val_dataset, checkpoint_folder, num_epochs=10, batch_size=32,
                learning_rate=0.001):
    """
    Train the model using the provided datasets.

    Args:
    - model: The model to be trained
    - train_dataset: Dataset for training
    - val_dataset: Dataset for validation
    - checkpoint_folder: Folder to store checkpoints
    - num_epochs: Number of epochs for training
    - batch_size: Batch size for training
    - learning_rate: Learning rate for optimization

    Returns:
    - model: Trained model
    - train_losses: List of training losses
    - val_losses: List of validation losses
    """
    # Create the checkpoint folder if it doesn't exist
    if not os.path.exists(checkpoint_folder):
        os.makedirs(checkpoint_folder)
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    print(f"Device: {device}")
    # Define data loaders for training and validation
    train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True, drop_last=True)
    val_loader = DataLoader(val_dataset, batch_size=batch_size, shuffle=False)

    # Define loss function and optimizer
    criterion = torch.nn.CosineEmbeddingLoss()
    optimizer = optim.SGD([{'params': model.cnn1.conv1.parameters()},
                        {'params': model.cnn1.layer1.parameters()},
                        {'params': model.cnn1.layer2.parameters()},
                        {'params': model.cnn1.layer3.parameters()},
                        {'params': model.cnn1.layer4.parameters()},
                        {'params': model.cnn1.fc.parameters(), 'lr':learning_rate*10}], lr=learning_rate, momentum=0.9)
    scheduler = lr_scheduler.StepLR(optimizer, step_size=10, gamma=0.5)

    # Lists to store training and validation losses
    train_losses = []
    val_losses = []

    # Variables to keep track of the best model and its performance
    best_val_loss = float('inf')
    best_model_state = None

    model = model.to(device)
    print("Training started...")
    for epoch in range(num_epochs):
        torch.cuda.empty_cache()
        print("*" * 100)
        print(f"Epoch [{epoch + 1}/{num_epochs}]:")
        model.train()
        running_train_loss = 0.0
        for i, (inputs, labels, _) in enumerate(train_loader):
            optimizer.zero_grad()
            # Forward pass
            inputA = inputs[0].to(device)
            inputB = inputs[1].to(device)
            labels = labels.to(device)
            output1, output2 = model(inputA, inputB)
            # Compute loss
            loss = criterion(output1, output2, labels)
            # Backward pass
            loss.backward()
            optimizer.step()
            running_train_loss += loss.item()

            if i % 200 == 0:
                print(f"\t Batch [{i}/{len(train_loader)}], Train Loss: {loss.item()}")

        # Compute average training loss for the epoch
        epoch_train_loss = running_train_loss / len(train_loader)
        train_losses.append(epoch_train_loss)

        # Validation loop
        model.eval()
        running_val_loss = 0.0
        with torch.no_grad():
            for i, (inputs, labels, _) in enumerate(val_loader):
                inputA = inputs[0].to(device)
                inputB = inputs[1].to(device)
                labels = labels.to(device)
                output1, output2 = model(inputA, inputB)
                loss = criterion(output1, output2, labels)
                running_val_loss += loss.item()

                if i % 25 == 0:
                    print(
                        f"Epoch [{epoch + 1}/{num_epochs}], Validation Batch [{i}/{len(val_loader)}], Val Loss: {loss.item()}")

        # Compute average validation loss for the epoch
        epoch_val_loss = running_val_loss / len(val_loader)
        val_losses.append(epoch_val_loss)

        # Save the model checkpoint for every epoch (last model)
        torch.save({
            'epoch': epoch,
            'model_state_dict': model.state_dict(),
            'optimizer_state_dict': optimizer.state_dict(),
            'val_loss': epoch_val_loss
        }, os.path.join(checkpoint_folder, f'last.pt'))

        # Save the best model checkpoint based on validation loss
        if epoch_val_loss < best_val_loss:
            best_val_loss = epoch_val_loss
            best_model_state = model.state_dict()
            print("Save the best checkpoint at epoch {}".format(epoch+1))
            torch.save({
                'epoch': epoch,
                'model_state_dict': best_model_state,
                'optimizer_state_dict': optimizer.state_dict(),
                'val_loss': best_val_loss
            }, os.path.join(checkpoint_folder, f'best.pt'))

        # Print progress
        print(f"Validation, Train Loss: {epoch_train_loss}, Val Loss: {epoch_val_loss}")
        print("*" * 100)
        scheduler.step()
    print("Training completed.")

    return model, train_losses, val_losses


# 5. Experiments

In [7]:
import datetime
now = datetime.datetime.now()

config = {
    "annotation_data_path":"/kaggle/input/mammo-224-224-ver2/split_data.csv",
    "image_folder_path": "/kaggle/input/mammo-dataset-ver4/archive/Processed_Images_450_200",
    "data_length": 256,
    "learning_rate":1e-3,
    "num_epoch": 12,
    "batch_size": 256,
    "checkpoint": "",
    "checkpoint_folder": f"/kaggle/working/resnet50BasedModel_{now}"
}

In [ ]:
import yaml
train_dataset = MammoCompDataset(data_path = config["image_folder_path"],
                                metadata = config["annotation_data_path"],
                                phase = "training",
                                datalen = config["data_length"],
                                seed=0)
valid_dataset = MammoCompDataset(data_path = config["image_folder_path"],
                                metadata = config["annotation_data_path"],
                                phase = "valid",
                                datalen = 1000,
                                seed=0)

model = SiameseNetwork101()

if config["checkpoint"]:
    checkpoint = torch.load(config["checkpoint"])
    model.load_state_dict(checkpoint["model_state_dict"])


train_model(model=model, train_dataset=train_dataset,
            val_dataset=valid_dataset, num_epochs=config["num_epoch"],
            batch_size=config["batch_size"], learning_rate=config["learning_rate"],
            checkpoint_folder=config["checkpoint_folder"]
            )

Downloading: "https://download.pytorch.org/models/resnet50-11ad3fa6.pth" to /root/.cache/torch/hub/checkpoints/resnet50-11ad3fa6.pth
100%|██████████| 97.8M/97.8M [00:03<00:00, 33.6MB/s]


Device: cpu
Training started...
****************************************************************************************************
Epoch [1/12]:
tensor([-1,  1, -1, -1,  1,  1,  1,  1, -1, -1, -1, -1,  1,  1,  1,  1,  1, -1,
        -1, -1, -1, -1,  1,  1,  1,  1,  1, -1, -1, -1,  1, -1, -1,  1,  1,  1,
         1,  1, -1, -1,  1,  1, -1, -1, -1,  1,  1,  1,  1, -1,  1, -1,  1, -1,
         1, -1,  1, -1,  1,  1, -1, -1,  1,  1, -1,  1, -1, -1,  1,  1,  1,  1,
         1, -1, -1,  1,  1, -1, -1, -1, -1,  1,  1, -1, -1, -1, -1, -1,  1, -1,
        -1,  1, -1,  1,  1, -1, -1,  1, -1, -1,  1,  1,  1,  1, -1, -1, -1, -1,
        -1,  1,  1,  1, -1,  1,  1,  1,  1, -1, -1, -1, -1,  1,  1, -1,  1,  1,
        -1, -1,  1,  1,  1, -1,  1,  1,  1, -1, -1,  1, -1,  1, -1,  1,  1, -1,
        -1,  1, -1, -1, -1, -1,  1, -1, -1, -1,  1, -1, -1,  1,  1,  1, -1,  1,
         1, -1,  1, -1, -1, -1, -1,  1,  1, -1, -1,  1, -1, -1, -1, -1,  1, -1,
         1, -1,  1, -1, -1,  1, -1, -1,  1, -1,  1, -